# <h1 style="color: #333333; background-color: #E6E6FA; text-align: center; padding: 10px; font-family: Arial;">DATA</h1>

## <h3 style="color: #333333; background-color: #b0cece; text-align: center; padding: 6px; font-family: 'Arial';">Libraries Importation</h3>

In [19]:
import os
import sys
import warnings

import sys
sys.path.append('../')

# Suppress warnings
warnings.filterwarnings('ignore')

### Basic Libraries
import numpy as np
import pandas as pd

### Sklearn
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

In [26]:
#FUNÇOES

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import numpy as np
from PIL import Image
import urllib
import seaborn as sns
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import silhouette_score, confusion_matrix
from sklearn.cluster import KMeans, AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage
import umap
import math

def set_plot_properties(ax, x_label, y_label, y_lim=[]):
    """
    Set properties of a plot axis.

    Args:
        ax (matplotlib.axes.Axes): The axis object of the plot.
        x_label (str): The label for the x-axis.
        y_label (str): The label for the y-axis.
        y_lim (list, optional): The limits for the y-axis. Defaults to [].

    Returns:
        None
    """
    ax.set_xlabel(x_label)  # Set the label for the x-axis
    ax.set_ylabel(y_label)  # Set the label for the y-axis
    if len(y_lim) != 0:
        ax.set_ylim(y_lim)  # Set the limits for the y-axis if provided


def plot_inertia_and_silhouette(data, k_min=2, k_max=15):
    """
    Plot the inertia (dispersion) and silhouette score for different numbers of clusters.

    Args:
        data (numpy.ndarray or pandas.DataFrame): The input data for clustering.
        k_min (int, optional): The minimum number of clusters to evaluate. Defaults to 2.
        k_max (int, optional): The maximum number of clusters to evaluate. Defaults to 15.

    Returns:
        None
    """
    dispersions = []
    scores = []

    k_clusters = range(k_min, k_max + 1)

    for k in k_clusters:
        kmeans = KMeans(n_clusters=k, random_state=0).fit(data)
        dispersions.append(kmeans.inertia_)  # Calculate the dispersion (inertia) for each number of clusters
        kmeans.predict(data)
        scores.append(silhouette_score(data, kmeans.labels_, metric='euclidean'))  # Calculate the silhouette score

    fig, (ax1, ax2) = plt.subplots(1, 2)
    ax1.plot(k_clusters, dispersions, marker='o')  # Plot the inertia (dispersion)
    set_plot_properties(ax1, 'Number of clusters', 'Dispersion (inertia)')
    ax2.plot(k_clusters, scores, marker='o')  # Plot the silhouette score
    set_plot_properties(ax2, 'Number of clusters', 'Silhouette score')

    plt.show()


def plot_dendrogram(data, linkage_method, cut_line=None):
    """
    Plot a dendrogram for hierarchical clustering.

    Args:
        data (numpy.ndarray or pandas.DataFrame): The input data for clustering.
        linkage_method (str): The linkage method used for clustering.
        cut_line (float, optional): The threshold value to cut the dendrogram. Defaults to None.

    Returns:
        None
    """
    # Fit the AgglomerativeClustering model
    model = AgglomerativeClustering(linkage=linkage_method, distance_threshold=0, n_clusters=None).fit(data)

    # Create the plot
    fig, ax = plt.subplots()
    plt.title('Hierarchical Clustering Dendrogram')

    # Create the counts of samples under each node
    counts = np.zeros(model.children_.shape[0])
    n_samples = len(model.labels_)

    for i, merge in enumerate(model.children_):
        current_count = 0
        for child_idx in merge:
            if child_idx < n_samples:
                current_count += 1  # Leaf node
            else:
                current_count += counts[child_idx - n_samples]
        counts[i] = current_count

    # Create the linkage matrix
    linkage_matrix = np.column_stack([model.children_, model.distances_, counts]).astype(float)

    # Plot the dendrogram
    dendrogram(linkage_matrix, truncate_mode='level', p=50)

    # Add a cut line if provided
    if cut_line is not None:
        plt.axhline(y=cut_line, color='black', linestyle='-')

    # Display the plot
    plt.show()


def plot_umap_projections(people, cluster_columns, n_components=2, random_state=42):
    """
    Generates UMAP projections for multiple clustering methods and plots them in a grid.

    Parameters:
    people (DataFrame): The input data containing the features and cluster assignments.
    cluster_columns (list of str): The list of column names containing cluster assignments for each method.
    n_components (int): Number of dimensions for UMAP projection. Default is 2.
    random_state (int): Random state for UMAP. Default is 42.
    """
    # Check if at least one cluster column is provided
    if len(cluster_columns) < 1:
        raise ValueError("At least one cluster column must be provided.")

    # Determine the layout of subplots
    num_plots = len(cluster_columns)
    num_rows = math.ceil(math.sqrt(num_plots))
    num_cols = math.ceil(num_plots / num_rows)

    # Create the subplots
    fig, axs = plt.subplots(num_rows, num_cols, figsize=(16, 12))
    axs = axs.flatten()  # Flatten the array of axes

    for i, cluster_column in enumerate(cluster_columns):
        # Check if the necessary columns are present in the dataframe
        if cluster_column not in people.columns:
            raise ValueError(f"Cluster column '{cluster_column}' not found in the data.")

        # Separate features and clusters
        features = people.drop(columns=[cluster_column])
        clusters = people[cluster_column]

        # Perform UMAP
        reducer = umap.UMAP(n_components=n_components, random_state=random_state)
        umap_embedding = reducer.fit_transform(features)

        # Plot the UMAP embedding
        scatter = axs[i].scatter(
            umap_embedding[:, 0], umap_embedding[:, 1],
            c=clusters, cmap='Spectral', s=5
        )
        axs[i].set_title(f'UMAP projection of {cluster_column}')
        axs[i].set_xlabel('UMAP 1')
        axs[i].set_ylabel('UMAP 2')

        # Add a color bar to each subplot
        cbar = plt.colorbar(scatter, ax=axs[i], boundaries=np.arange(clusters.nunique() + 1) - 0.5)
        cbar.set_ticks(np.arange(clusters.nunique()))

    # Remove any unused subplots
    for j in range(i + 1, len(axs)):
        fig.delaxes(axs[j])

    # Adjust layout
    plt.tight_layout()
    plt.show()


def groupby_mean(data, variable, n_features=30):
    """
    Group the data by a variable and calculate the mean for each group.

    Args:
        data (pandas.DataFrame): The input data.
        variable (str): The variable used for grouping.
        n_features (int, optional): The number of features to include in the result. Defaults to 30.

    Returns:
        pandas.DataFrame: The transposed DataFrame containing the mean values for each group.
    """
    # Group the data by the specified variable and calculate the mean for each group
    grouped_data = data.groupby(variable).mean()

    # Select the first n_features + 1 columns (including the variable column) and transpose the DataFrame
    result = grouped_data.iloc[:, :n_features + 1].T

    # Return the transposed DataFrame
    return result

## <h3 style="color: #333333; background-color: #b0cece; text-align: center; padding: 6px; font-family: 'Arial';">Data Importation</h3>

In [2]:
people = pd.read_csv('testdata.csv', index_col = 'Loyalty#')

In [3]:
people

,Unnamed: 0.1,Unnamed: 0,First Name,Last Name,Customer Name,Country,Province or State,City,Latitude,Longitude,...,Total_Spend_Month_7,Total_Spend_Month_8,Total_Spend_Month_9,Total_Spend_Month_10,Total_Spend_Month_11,Total_Spend_Month_12,TotalFlightsWithCompanions,TotalFlightSpend,PctFlightsWithCompanions,PctPointsSpent
Loyalty#,,,,,,,,,,,,,,,,,,,,,
480934,0,0,Cecilia,Householder,Cecilia Householder,Canada,Ontario,Toronto,43.653225,-79.383186,...,0.0,0.0,0.0,37.0,45.6,52.0,56.0,134.6,0.292276,0.266628
549612,1,1,Dayle,Menez,Dayle Menez,Canada,Alberta,Edmonton,53.544388,-113.490930,...,195.4,0.0,0.0,0.0,0.0,0.0,32.0,221.4,0.104541,0.526282
429460,2,2,Necole,Hannon,Necole Hannon,Canada,British Columbia,Vancouver,49.282730,-123.120740,...,0.0,0.0,0.0,0.0,0.0,0.0,40.0,53.2,0.325203,0.229922
608370,3,3,Queen,Hagee,Queen Hagee,Canada,Ontario,Toronto,43.653225,-79.383186,...,0.0,0.0,0.0,0.0,0.0,33.0,58.0,162.2,0.296069,0.423144
530508,4,4,Claire,Latting,Claire Latting,Canada,Quebec,Hull,45.428730,-75.713364,...,0.0,0.0,0.0,0.0,0.0,0.0,63.0,0.0,0.364794,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100012,16752,15,Ethan,Thompson,Ethan Thompson,Canada,Quebec,Quebec City,46.759733,-71.141009,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
100013,16753,16,Layla,Young,Layla Young,Canada,Alberta,Edmonton,53.524829,-113.546357,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
100014,16754,17,Amelia,Bennett,Amelia Bennett,Canada,New Brunswick,Moncton,46.051866,-64.825428,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
people = people.loc[:, ~people.columns.str.contains('^Unnamed')]

In [10]:
cols_to_drop = [
    "Unnamed: 0.1",
    "Unnamed: 0",
    "First Name",
    "Last Name",
    "Customer Name",
    "Province or State",
    "City",
    "Latitude",
    "Longitude",
    "Postal code",
    "Country",
    "Location Code",
    "EnrollmentDateOpening",
    "CancellationDate",
    "EnrollmentType",
    "LastFlightDate"
]

people = people.drop(columns=cols_to_drop, errors="ignore")


In [6]:
# --- GENDER BINARY ENCODING ---
# Convert to lowercase and strip spaces just in case
people["Gender"] = people["Gender"].str.lower().str.strip()

# Map to binary (you can customize)
gender_map = {
    "male": 1,
    "m": 1,
    "female": 0,
    "f": 0
}

people["Gender_Binary"] = people["Gender"].map(gender_map)

# If there are unknown values, fill with NaN or a special code
people["Gender_Binary"] = people["Gender_Binary"].fillna(-1)   # -1 = unknown


# --- ONE-HOT ENCODE LOYALTY STATUS ---
people = pd.get_dummies(people, columns=["LoyaltyStatus"], prefix="Loyalty", dtype=int)

# --- ONE-HOT ENCODE Education ---
people = pd.get_dummies(people, columns=["Education"], prefix="Education", dtype=int)

# --- ONE-HOT ENCODE MARITAL STATUS ---
people = pd.get_dummies(people, columns=["Marital Status"], prefix="Marital", dtype=int)


In [7]:
cols_to_drop1 = [
    "Gender",
]

people = people.drop(columns=cols_to_drop1, errors="ignore")

In [11]:
people

,Income,Customer Lifetime Value,TotalFlights,TotalDistance,TotalPointsAccumulated,TotalPointsRedeemed,RecencyInMonths,Pct_Spend_Month_1,Pct_Spend_Month_2,Pct_Spend_Month_3,...,Loyalty_Nova,Loyalty_Star,Education_Bachelor,Education_College,Education_Doctor,Education_High School or Below,Education_Master,Marital_Divorced,Marital_Married,Marital_Single
Loyalty#,,,,,,,,,,,,,,,,,,,,,
480934,70146.0,3839.14,191.6,507054.9,50699.39,13517.9,1.018397,0.049646,0.129181,0.024300,...,0,1,1,0,0,0,0,0,1,0
549612,0.0,3839.61,306.1,426827.4,42672.54,22457.8,1.018397,0.061351,0.061222,0.067454,...,0,1,0,1,0,0,0,1,0,0
429460,0.0,3839.75,123.0,238376.1,23832.41,5479.6,11.990802,0.280099,0.025545,0.127345,...,0,1,0,1,0,0,0,0,0,1
608370,0.0,3839.75,195.9,386029.3,38595.63,16331.5,1.018397,0.127599,0.091180,0.170102,...,0,1,0,1,0,0,0,0,0,1
530508,97832.0,3842.79,172.7,369242.6,36916.56,0.0,1.018397,0.000000,0.053402,0.022173,...,0,1,1,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100012,34148.0,5780.18,0.0,0.0,0.00,0.0,999.000000,NaN,NaN,NaN,...,0,1,1,0,0,0,0,0,0,1
100013,34148.0,5780.18,0.0,0.0,0.00,0.0,999.000000,NaN,NaN,NaN,...,0,1,1,0,0,0,0,0,1,0
100014,34148.0,5780.18,0.0,0.0,0.00,0.0,999.000000,NaN,NaN,NaN,...,0,1,1,0,0,0,0,0,1,0


In [12]:
people.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16757 entries, 480934 to 100016
Data columns (total 47 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Income                          16757 non-null  float64
 1   Customer Lifetime Value         16757 non-null  float64
 2   TotalFlights                    16757 non-null  float64
 3   TotalDistance                   16757 non-null  float64
 4   TotalPointsAccumulated          16757 non-null  float64
 5   TotalPointsRedeemed             16757 non-null  float64
 6   RecencyInMonths                 16757 non-null  float64
 7   Pct_Spend_Month_1               16737 non-null  float64
 8   Pct_Spend_Month_2               16737 non-null  float64
 9   Pct_Spend_Month_3               16737 non-null  float64
 10  Pct_Spend_Month_4               16737 non-null  float64
 11  Pct_Spend_Month_5               16737 non-null  float64
 12  Pct_Spend_Month_6              

In [13]:
people1 = people.copy()

In [14]:
for column in people1.columns:
    if column.startswith('spend'):
        people1[column] = np.sqrt(people1[column])

In [15]:
people_st_scl = StandardScaler().fit_transform(people1)
people_mm_scl = MinMaxScaler().fit_transform(people1)
people_rb_scl = RobustScaler().fit_transform(people1)

# <h1 style="color: #333333; background-color: #E6E6FA; text-align: center; padding: 10px; font-family: Arial;">MODELING</h1>

## <h3 style="color: #333333; background-color: #b0cece; text-align: center; padding: 6px; font-family: 'Arial';">Standard Scaling</h3>

### K-Means

In [16]:
plot_inertia_and_silhouette(people_st_scl)

NameError: name 'plot_inertia_and_silhouette' is not defined

In [ ]:
kmeans_st_scl = KMeans(n_clusters=7, random_state=42).fit(people_st_scl)
people['st_scaled_km_cluster7'] = kmeans_st_scl.predict(people_st_scl)

In [ ]:
groupby_mean(people, 'st_scaled_km_cluster7')

### Hierarchichal - Ward

In [ ]:
plot_dendrogram(people_st_scl, 'ward')

In [ ]:
people['st_scaled_ward_cluster7'] = AgglomerativeClustering(
    linkage = 'ward', n_clusters = 7
    ).fit_predict(people_st_scl)

In [ ]:
groupby_mean(people, 'st_scaled_ward_cluster7')

## <h3 style="color: #333333; background-color: #b0cece; text-align: center; padding: 6px; font-family: 'Arial';">MinMax Scaling</h3>

### K-Means

In [17]:
plot_inertia_and_silhouette(people_mm_scl)

NameError: name 'plot_inertia_and_silhouette' is not defined

In [ ]:
kmeans_mm_scl = KMeans(n_clusters=7, random_state=42).fit(people_mm_scl)
people['mm_scaled_km_cluster7'] = kmeans_mm_scl.predict(people_mm_scl)

In [ ]:
groupby_mean(people, 'mm_scaled_km_cluster7')

### Hierarchichal - Ward

In [ ]:
plot_dendrogram(people_mm_scl, 'ward')

In [ ]:
people['mm_scaled_ward_cluster7'] = AgglomerativeClustering(
    linkage = 'ward', n_clusters = 7
    ).fit_predict(people_mm_scl)

In [ ]:
groupby_mean(people, 'mm_scaled_ward_cluster7')

## <h3 style="color: #333333; background-color: #b0cece; text-align: center; padding: 6px; font-family: 'Arial';">Robust Scaling</h3>

### K-Means

In [18]:
plot_inertia_and_silhouette(people_rb_scl)

NameError: name 'plot_inertia_and_silhouette' is not defined

In [ ]:
kmeans_rb_scl = KMeans(n_clusters=7, random_state=42).fit(people_rb_scl)
people['rb_scaled_km_cluster7'] = kmeans_rb_scl.predict(people_rb_scl)

In [ ]:
groupby_mean(people, 'rb_scaled_km_cluster7')

### Hierarchichal - Ward

In [ ]:
plot_dendrogram(people_rb_scl, 'ward')

In [ ]:
people['rb_scaled_ward_cluster7'] = AgglomerativeClustering(
    linkage = 'ward', n_clusters = 7
    ).fit_predict(people_rb_scl)

In [ ]:
groupby_mean(people, 'rb_scaled_ward_cluster7')

# <h1 style="color: #333333; background-color: #E6E6FA; text-align: center; padding: 10px; font-family: Arial;">FILTERING</h1>

### <h3 style="color: #333333; background-color: #b0cece; text-align: center; padding: 6px; font-family: 'Arial';">(1) First-Examination</h3>

#### Cluster Visualization 1

In [ ]:
clusters_km = ['st_scaled_km_cluster7', 'mm_scaled_km_cluster7', 'rb_scaled_km_cluster7']

In [ ]:
plot_umap_projections(people, clusters_km)

In [ ]:
cluster_ward = ['st_scaled_ward_cluster7', 'mm_scaled_ward_cluster7', 'rb_scaled_ward_cluster7']

In [ ]:
plot_umap_projections(people, cluster_ward)

In [ ]:
groupby_mean(people, 'rb_scaled_ward_cluster7')

5(veterinaries) \ 1(Vegetarian) \ 5(Gamers) \ 7(loyals) \ 2(?) , 3(promotion) , 4(alcoholics) , 6(parents) , 8(young adults)

#### Cluster Extraction 1

In [ ]:
# Filter the DataFrame based on cluster labels
veterinaries = people[people['rb_scaled_ward_cluster7'] == 5]
vegetarians = people[people['rb_scaled_ward_cluster7'] == 4]
gamers = people[people['rb_scaled_ward_cluster7'] == 3]
loyals = people[people['rb_scaled_ward_cluster7'] == 6]

In [ ]:
# List of cluster labels to exclude
exclude_clusters = [5, 4, 3, 6]

# Filter the original dataset to exclude rows with specified cluster labels
people_filtered1 = people[~people['rb_scaled_ward_cluster7'].isin(exclude_clusters)]

In [ ]:
people_filtered1

In [ ]:
people_filtered1 = people_filtered1.iloc[:, :-6]

### <h3 style="color: #333333; background-color: #b0cece; text-align: center; padding: 6px; font-family: 'Arial';">(2) Second-Examination</h3>

#### Re-Scaling Data 2

In [ ]:
people_filtered1

In [ ]:
people_filtered11 = people_filtered1.copy()

for column in people_filtered11.columns:
    if column.startswith('spend'):
        people_filtered11[column] = np.sqrt(people_filtered11[column])

In [ ]:
people_filtered1_st_scl = StandardScaler().fit_transform(people_filtered11)
people_filtered1_rb_scl = RobustScaler().fit_transform(people_filtered11)

#### Cluster Modelling 2

In [ ]:
plot_inertia_and_silhouette(people_filtered1_st_scl)

In [ ]:
plot_inertia_and_silhouette(people_filtered1_rb_scl)

In [ ]:
plot_dendrogram(people_filtered1_st_scl, 'ward')

In [ ]:
plot_dendrogram(people_filtered1_rb_scl, 'ward')

#### Cluster Visualization 2

In [ ]:
people_filtered1['filter1_rb_scaled_ward_cluster3'] = AgglomerativeClustering(
    linkage = 'ward', n_clusters = 3
    ).fit_predict(people_filtered1_rb_scl)

people_filtered1['filter1_st_scaled_ward_cluster2'] = AgglomerativeClustering(
    linkage = 'ward', n_clusters = 2
    ).fit_predict(people_filtered1_st_scl)

In [ ]:
kmeans_rb_scl = KMeans(n_clusters=5, random_state=42).fit(people_filtered1_rb_scl)
people_filtered1['filter1_rb_scaled_km_cluster5'] = kmeans_rb_scl.predict(people_filtered1_rb_scl)

kmeans_st_scl = KMeans(n_clusters=5, random_state=42).fit(people_filtered1_st_scl)
people_filtered1['filter1_st_scaled_km_cluster5'] = kmeans_st_scl.predict(people_filtered1_st_scl)

In [ ]:
cluster_filter1 = ['filter1_rb_scaled_ward_cluster3', 'filter1_st_scaled_ward_cluster2', 'filter1_rb_scaled_km_cluster5', 'filter1_st_scaled_km_cluster5']

In [ ]:
plot_umap_projections(people_filtered1, cluster_filter1)

In [ ]:
groupby_mean(people_filtered1, 'filter1_st_scaled_km_cluster5')


# CORRER OS OUTROS GROUP BY
# FAZER LABEL DOS CLUSTERS PRESENTES

In [ ]:
# 3 - alcoholics | 1 - parents | 2 - young adults technology

#### Cluster Extraction 2

In [ ]:
# Filter the DataFrame based on cluster labels
alcoholics = people_filtered1[people_filtered1['filter1_st_scaled_km_cluster5'] == 3]

In [ ]:
# List of cluster labels to exclude
exclude_clusters = [3]

# Filter the original dataset to exclude rows with specified cluster labels
people_filter2 = people_filtered1[~people_filtered1['filter1_st_scaled_km_cluster5'].isin(exclude_clusters)]

In [ ]:
people_filter2 = people_filter2.iloc[:, :-4]

# <h1 style="color: #333333; background-color: #E6E6FA; text-align: center; padding: 10px; font-family: Arial;">CLUSTER EXPORTATION</h1>

In [ ]:
# Define file names for CSV files
file_names = ['veterinaries.csv', 'vegetarians.csv', 'gamers.csv', 'loyals.csv', 'alcoholics.csv', 'families.csv', 'young_adults.csv', 'karens.csv', 'promo_hunters.csv']

# Define dataframes and filenames
datasets = [veterinaries, vegetarians, gamers, loyals, alcoholics, families, young_adults, karens, promo_hunters]

# Save each subset as a CSV file
for df, file_name in zip(datasets, file_names):
    df.to_csv(file_name)